# Safe Lag Feature Engineering (v2)

Pipeline thay thế lag ngắn (lag_1/2/3/7, rolling_mean_7/14) bằng các feature an toàn cho 16-ngày forecast horizon:

| Feature mới | Lookback an toàn (Aug 16–31) |
|---|---|
| `lag_16` | d−16 ≤ Aug 15 ✓ |
| `rolling_mean_30` | shift(16).rolling(30) → d−45 đến d−16 ✓ |

**Kaggle test period: 2017-08-16 → 2017-08-31 (16 ngày)**

## 1. Setup & Load Data

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
PROJECT_ROOT = next(
    (p for p in [_here] + list(_here.parents) if (p / 'config.py').exists()),
    _here
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import PROJECT_ROOT, DATA_DIR, RAW_DIR, PROCESSED_DIR, SPLITS_DIR

MODELS_DIR    = PROJECT_ROOT / 'models'
NOTEBOOKS_DIR = PROJECT_ROOT / 'notebooks'

print(f'PROJECT_ROOT : {PROJECT_ROOT}')
print(f'PROCESSED_DIR: {PROCESSED_DIR}')
print(f'SPLITS_DIR   : {SPLITS_DIR}')

In [ ]:
import pandas as pd
import numpy as np
import shutil

# ── Load full raw training data (2013-01-01 to 2017-08-15) ──────────────────
print('Loading train_cleaned.csv  (3M rows × 6 cols)...')
train_raw = pd.read_csv(PROCESSED_DIR / 'cleaned' / 'train_cleaned.csv')
train_raw['date'] = pd.to_datetime(train_raw['date'])
train_raw = train_raw.sort_values(['store_nbr', 'family', 'date']).reset_index(drop=True)

print(f'train_raw : {train_raw.shape}')
print(f'date range: {train_raw["date"].min().date()} → {train_raw["date"].max().date()}')
print(f'columns   : {train_raw.columns.tolist()}')
print(f'sales stats: min={train_raw["sales"].min():.2f}, max={train_raw["sales"].max():.2f}, mean={train_raw["sales"].mean():.2f}')

## 2. Compute Safe Lag Features

### Tại sao lag_1–7 độc hại?
Khi forecast 16 ngày (Aug 16–31) không có dữ liệu thật:
- `lag_1` tại Aug 17 cần Aug 16 → phải dùng *prediction* → **error propagation**
- `lag_7` tại Aug 22 cần Aug 15 (OK) nhưng Aug 23 cần Aug 16 → prediction

### Tại sao lag_16 & rolling_mean_30 an toàn?
- `lag_16` tại Aug 31 cần Aug 15 → **trong train data thật** ✓
- `rolling_mean_30 = shift(16).rolling(30)` → lookback d−45 đến d−16 ≤ Aug 15 ✓

In [ ]:
print('Computing new safe lag features...')

grouped = train_raw.groupby(['store_nbr', 'family'], sort=False)['sales']

# lag_16: sales từ 16 ngày trước
train_raw['lag_16'] = grouped.shift(16)

# rolling_mean_30: mean của sales từ lag_16 đến lag_45
# Công thức: x.shift(16).rolling(30, min_periods=1).mean()
# Tại vị trí i: mean(sales[i-45], sales[i-44], ..., sales[i-16])
train_raw['rolling_mean_30'] = grouped.transform(
    lambda x: x.shift(16).rolling(30, min_periods=1).mean()
)

print('\nNaN summary trên full dataset (3M rows):')
for col in ['lag_16', 'rolling_mean_30']:
    n   = train_raw[col].isnull().sum()
    pct = n / len(train_raw) * 100
    print(f'  {col:<20}: {n:>7,} NaN ({pct:.3f}%)')

print('\nSample (store 1, AUTOMOTIVE, first 50 rows):')
sample = train_raw[(train_raw['store_nbr']==1) & (train_raw['family']=='AUTOMOTIVE')].head(50)
print(sample[['date','sales','lag_16','rolling_mean_30']].tail(10).to_string())

## 3. Xác minh An toàn cho Test Periods

In [ ]:
print('=== VERIFICATION: Internal test period (2017-08-01 → 2017-08-15) ===\n')
test_mask    = (train_raw['date'] >= '2017-08-01') & (train_raw['date'] <= '2017-08-15')
test_subset  = train_raw[test_mask]

for col in ['lag_16', 'rolling_mean_30']:
    n = test_subset[col].isnull().sum()
    status = 'PASS ✓' if n == 0 else f'FAIL ✗ ({n} NaN)'
    print(f'  {col:<20}: {status}')
    assert n == 0, f'BUG: {n} NaN trong {col} ở internal test period!'

print()
print('=== Lookback trace cho Kaggle test (2017-08-16 → 2017-08-31) ===\n')
print(f'{"Date":<12} {"lag_16 needs":<18} {"In train?":<12} {"rm30 needs":<22} {"In train?"}' )
for day in range(16, 32):
    d         = pd.Timestamp(f'2017-08-{day:02d}')
    l16_date  = d - pd.Timedelta(days=16)
    rm30_from = d - pd.Timedelta(days=45)
    rm30_to   = d - pd.Timedelta(days=16)
    train_end = pd.Timestamp('2017-08-15')
    l16_ok    = l16_date  <= train_end
    rm30_ok   = rm30_to   <= train_end
    print(
        f'Aug {day:<4}   {l16_date.strftime("%Y-%m-%d"):<18}'
        f'{"✓" if l16_ok else "✗":<12}'
        f'{rm30_from.strftime("%Y-%m-%d")} – {rm30_to.strftime("%Y-%m-%d"):<10}'
        f'{"✓" if rm30_ok else "✗"}'
    )

print('\nAll lag_16 and rolling_mean_30 lookbacks are within training data ✓')

## 4. Update Existing Splits

Load splits cũ → xóa 6 feature độc hại → thêm 2 feature mới → lưu vào `splits_safe_lag/`.

In [ ]:
TOXIC_FEATURES = ['lag_1', 'lag_2', 'lag_3', 'lag_7', 'rolling_mean_7', 'rolling_mean_14']

print(f'Loading existing splits from: {SPLITS_DIR}')
X_train = pd.read_csv(SPLITS_DIR / 'train_features.csv')
X_val   = pd.read_csv(SPLITS_DIR / 'val_features.csv')
X_test  = pd.read_csv(SPLITS_DIR / 'test_features.csv')

print(f'\nOriginal shapes:')
print(f'  X_train : {X_train.shape}   (rows: 2013-01-01 → 2017-06-30)')
print(f'  X_val   : {X_val.shape}    (rows: 2017-07-01 → 2017-07-31)')
print(f'  X_test  : {X_test.shape}   (rows: 2017-08-01 → 2017-08-15)')

# Kiểm tra toxic features tồn tại
for col in TOXIC_FEATURES:
    assert col in X_train.columns, f'Toxic feature not found: {col}'
print(f'\nVerified: all {len(TOXIC_FEATURES)} toxic features present in splits')

In [ ]:
# ── Xóa toxic features ─────────────────────────────────────────────────────
X_train = X_train.drop(columns=TOXIC_FEATURES)
X_val   = X_val.drop(columns=TOXIC_FEATURES)
X_test  = X_test.drop(columns=TOXIC_FEATURES)
print(f'After removing {len(TOXIC_FEATURES)} toxic cols: X_train {X_train.shape}')

# ── Chuẩn bị new features từ train_raw ────────────────────────────────────
new_feats = train_raw[['date', 'store_nbr', 'family', 'lag_16', 'rolling_mean_30']].copy()
new_feats['date'] = new_feats['date'].dt.strftime('%Y-%m-%d')

# ── Merge vào từng split ──────────────────────────────────────────────────
before_rows = {name: df.shape[0] for name, df in [('train', X_train), ('val', X_val), ('test', X_test)]}

X_train = X_train.merge(new_feats, on=['date', 'store_nbr', 'family'], how='left')
X_val   = X_val.merge(new_feats,   on=['date', 'store_nbr', 'family'], how='left')
X_test  = X_test.merge(new_feats,  on=['date', 'store_nbr', 'family'], how='left')

# Đảm bảo row count không thay đổi
for name, df, before in [('train', X_train, before_rows['train']),
                          ('val',   X_val,   before_rows['val']),
                          ('test',  X_test,  before_rows['test'])]:
    assert len(df) == before, f'Row count changed in {name}: {before} → {len(df)}'

print(f'After adding lag_16, rolling_mean_30:')
print(f'  X_train : {X_train.shape}')
print(f'  X_val   : {X_val.shape}')
print(f'  X_test  : {X_test.shape}')
print(f'\nFull feature list ({X_train.shape[1]} cols):')
print(X_train.dtypes.to_string())

In [ ]:
print('=== NaN VERIFICATION sau khi merge ===\n')
for split_name, df in [('X_train', X_train), ('X_val', X_val), ('X_test', X_test)]:
    print(f'--- {split_name} ---')
    nan_counts = df[['lag_16', 'rolling_mean_30']].isnull().sum()
    for col, n in nan_counts.items():
        status = '✓ 0 NaN' if n == 0 else f'✗ {n} NaN'
        print(f'  {col:<20}: {status}')
    print()

# Assertion: X_test phải có 0 NaN trong lag_16 và rolling_mean_30
for col in ['lag_16', 'rolling_mean_30']:
    n = X_test[col].isnull().sum()
    assert n == 0, f'FAIL: {n} NaN trong {col} ở X_test!'
print('All assertions passed ✓')

print('\nSample X_test (last 5 rows):')
disp_cols = ['date', 'store_nbr', 'family', 'lag_14', 'lag_16', 'lag_28', 'lag_364',
             'rolling_mean_28', 'rolling_mean_30', 'rolling_std_7']
print(X_test[disp_cols].tail(5).to_string())

## 5. Save New Splits

In [ ]:
SPLITS_SAFE = PROCESSED_DIR / 'splits_safe_lag'
SPLITS_SAFE.mkdir(parents=True, exist_ok=True)

print(f'Saving to: {SPLITS_SAFE}')

# Feature files (modified)
X_train.to_csv(SPLITS_SAFE / 'train_features.csv', index=False)
X_val.to_csv(SPLITS_SAFE   / 'val_features.csv',   index=False)
X_test.to_csv(SPLITS_SAFE  / 'test_features.csv',  index=False)

# Target files (unchanged — copy from original splits)
for fname in ['train_target.csv', 'val_target.csv', 'val_target_original.csv', 'test_target.csv']:
    shutil.copy(SPLITS_DIR / fname, SPLITS_SAFE / fname)
    print(f'  Copied : {fname}')

print(f'\n  Saved  : train_features.csv  {X_train.shape}')
print(f'  Saved  : val_features.csv    {X_val.shape}')
print(f'  Saved  : test_features.csv   {X_test.shape}')

# Verify saved files
for fname, expected_shape in [
    ('train_features.csv', X_train.shape),
    ('val_features.csv',   X_val.shape),
    ('test_features.csv',  X_test.shape),
]:
    saved = pd.read_csv(SPLITS_SAFE / fname, nrows=5)
    assert saved.shape[1] == expected_shape[1], f'Column count mismatch in {fname}'
print('\nSave verification: ✓')
print(f'\nFeature count: {X_train.shape[1]} (was 49, removed 6 toxic, added 2 safe → net -4)')
print(f'Toxic removed: {TOXIC_FEATURES}')
print(f'New features : [lag_16, rolling_mean_30]')